In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from collections import defaultdict
import copy
import os
import pprint
import functools
from pathlib import Path

import hydra
import duckdb
from omegaconf import OmegaConf
from einops import rearrange
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches
import matplotlib.colors
import seaborn as sns
font_scale = 7
sns.set_theme(style='ticks', font_scale=font_scale, palette=sns.color_palette('Set2'),)
import sqlalchemy as sa
import marimo as mo
import seaborn as sns
import lightning.pytorch
import polars as pl
from tqdm import tqdm
import torch
from great_tables import GT

from conf import conf
from dafm import datasets, models, plots, utils

In [17]:
duckdb.sql("""
attach '../runs.sqlite';
use runs;
""")

BinderException: Binder Error: Failed to attach database: database with name "runs" already exists

In [ ]:
engine = conf.get_engine()
session = conf.sa.orm.Session(engine)
session.begin()

# Run queries

### Datasets

In [ ]:
dataset_cols = """
    Dataset.id,
    time_step_count,
    time_step_count_drop_first,
    observe_every_n_time_steps,
"""

In [18]:
dataset_info = pl.DataFrame(dict(
    dataset_name=['Lorenz96', 'KuramotoSivashinsky', 'NavierStokesDim64', 'NavierStokesDim256'],
    dataset_name_latex=["Lorenz '96", 'Kuramoto-Sivashinsky (1,024 dim)', r'Navier-Stokes ($64 \times 64$)', r'Navier-Stokes ($256 \times 256$)'],
    sampling_time_step_count=[5, 5, 10, 10],
    multiple=1,
))
dataset_info

dataset_name,dataset_name_latex,sampling_time_step_count,multiple
str,str,i64,i32
"""Lorenz96""","""Lorenz '96""",5,1
"""KuramotoSivashinsky""","""Kuramoto-Sivashinsky (1,024 di…",5,1
"""NavierStokesDim64""","""Navier-Stokes ($64 \times 64$)""",10,1
"""NavierStokesDim256""","""Navier-Stokes ($256 \times 256…",10,1


##### Lorenz '96

In [41]:
l96_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'Lorenz96'
    join Lorenz96 on Dataset.id = Lorenz96.id
    where true
    and state_dimension = 1e6
    and use_predicted_state_perturbation = false
    and predicted_state_perturbation_std = 0.
""")
assert len(l96_rows) == 1

##### Kuramoto-Sivashinsky

In [43]:
ks_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'KuramotoSivashinsky'
    join KuramotoSivashinsky on Dataset.id = KuramotoSivashinsky.id
    join ATan on Dataset.Observe = ATan.id
    where true
    and state_dimension = 1024
    and floating_point_precision = 32
    and use_predicted_state_perturbation = false
    and predicted_state_perturbation_std = 0.
    and predicted_state_count = 20
""")
assert len(ks_rows) == 1

##### Navier-Stokes ($64 \times 64$)

In [44]:
ns64_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'NavierStokesDim64'
    join NavierStokes on Dataset.id = NavierStokes.id
    join ATan on Dataset.Observe = ATan.id
    where true
    and state_dimension = 3*64*64
    and use_predicted_state_perturbation = false
    and predicted_state_perturbation_std = 0.
    and predicted_state_count = 20
""")
assert len(ns64_rows) == 1

##### Navier-Stokes ($256 \times 256$)

In [45]:
ns256_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'NavierStokesDim256'
    join NavierStokes on Dataset.id = NavierStokes.id
    where true
    and state_dimension = 3*256*256
    and time_step_count = 6000
    and time_step_size = 1e-4
    and observe_every_n_time_steps = 100
    and use_predicted_state_perturbation = false
    and predicted_state_perturbation_std = 0.
""").pl()
assert len(ns256_rows) == 1

### Models

In [74]:
sampling_time_step_counts = (5, 10, 20, 50, 100)

In [75]:
filepaths = duckdb.sql("""
    select
        dataset_name, label, format('{}/TopK_{}_T{}.csv', dataset_name, label, sampling_time_step_count) as filepath
    from dataset_info
    cross join (
        select * as label from (values ('EnSF'), ('EnFF-OT'), ('EnFF-F2P'))
    )
""").pl()
filepath_exists = []
for f in filepaths.get_column('filepath'):
    f = Path(f).expanduser()
    exists = f.exists()
    if not exists:
        print(f"File path does not exist: '{f}'")
    filepath_exists.append(exists)
filepaths = pl.DataFrame(dict(
    dataset_name=filepaths['dataset_name'],
    label=filepaths['label'],
    filepath=filepaths.get_column('filepath'),
    exists=filepath_exists,
))
assert filepaths['exists'].any()
duckdb.sql("""
set variable hyperparameter_filepaths = (
    select list(filepath) from filepaths where exists
)
""")
hyperparameters = duckdb.sql("""
    select
        filepaths.dataset_name,
        label,
        epsilon_alpha,
        epsilon_beta,
        sigma_min,
        lambda,
    from read_csv(getvariable(hyperparameter_filepaths), filename=true, union_by_name=true) as csv_data
    join filepaths
    on csv_data.filename = filepaths.filepath
    where k = 1
""").pl()
hyperparameters

dataset_name,label,epsilon_alpha,epsilon_beta,sigma_min,lambda
str,str,f64,f64,f64,f64
"""Lorenz96""","""EnSF""",1.0,0.275,null,null
"""KuramotoSivashinsky""","""EnSF""",1.0,0.275,null,null
"""NavierStokesDim64""","""EnSF""",1.0,0.275,null,null
"""NavierStokesDim256""","""EnSF""",1.0,0.275,null,null
"""Lorenz96""","""EnFF-OT""",null,null,0.1,0.05
…,…,…,…,…,…
"""NavierStokesDim256""","""EnFF-OT""",null,null,0.01,0.005
"""Lorenz96""","""EnFF-F2P""",null,null,0.0001,0.05
"""KuramotoSivashinsky""","""EnFF-F2P""",null,null,0.001,0.005


In [76]:
var = {
    'EnSF': [
        ['epsilon_alpha', r'$\epsilon_{\alpha}$'],
        ['epsilon_beta', r'$\epsilon_{\beta}$'],
    ],
    'EnFF-OT': [
        ['log10(sigma_min)', r'$\log_{10}(\sigma_{\min})$'],
        ['lambda', r'$\lambda$'],
    ],
}
var['EnFF-F2P'] = var['EnFF-OT']

##### EnsF

In [77]:
label = 'EnSF'
var1, var1_latex = var[label][0]
var2, var2_latex = var[label][1]
ensf_rows = duckdb.sql(f"""
    select
        Model.id,
        {label!r} as label,
        {var1} as var1,
        {var2} as var2,
        sampling_time_step_count,
    from Model
    join ScoreMatchingMarginal on Model.id = ScoreMatchingMarginal.id
    join Bao2024EnsembleScoreMatching on ScoreMatchingMarginal.DiffusionPath = Bao2024EnsembleScoreMatching.id
    where true
    and sampling_time_step_count in {sampling_time_step_counts}
    and (epsilon_alpha, epsilon_beta) in (select epsilon_alpha, epsilon_beta from hyperparameters where label = {label!r})
""")
ensf_multiple = (
    len(sampling_time_step_counts)
    * len(hyperparameters.filter(label=label)[['epsilon_alpha', 'epsilon_beta']].unique())
)
assert len(ensf_rows) == ensf_multiple

##### EnFF-OT

In [78]:
label = 'EnFF-OT'
var1, var1_latex = var[label][0]
var2, var2_latex = var[label][1]
enff_ot_rows = duckdb.sql(f"""
    select distinct
        Model.id,
        {label!r} as label,
        ConditionalOptimalTransport.sigma_min as var1,
        Constant.constant as var2,
        sampling_time_step_count,
    from Model
    join FlowMatchingMarginal on Model.id = FlowMatchingMarginal.id
    join ConditionalOptimalTransport on FlowMatchingMarginal.DiffusionPath = ConditionalOptimalTransport.id
    join Local on FlowMatchingMarginal.EnergyGuidance = Local.id
    join Constant on Local.Schedule = Constant.id
    join hyperparameters on (
            hyperparameters.label = {label!r}
        and
            (hyperparameters.sigma_min, hyperparameters.lambda) = (ConditionalOptimalTransport.sigma_min, Constant.constant)
    )
    where true
    and sampling_time_step_count in {sampling_time_step_counts}
""")
enff_ot_multiple = (
    len(sampling_time_step_counts)
    * len(hyperparameters.filter(label=label)[['sigma_min', 'lambda']].unique())
)
assert len(enff_ot_rows) == enff_ot_multiple

##### EnFF-F2P

In [79]:
label = 'EnFF-F2P'
var1, var1_latex = var[label][0]
var2, var2_latex = var[label][1]
enff_f2p_rows = duckdb.sql(f"""
    select distinct
        Model.id,
        {label!r} as label,
        PreviousPosteriorToPredictive.sigma_min as var1,
        Constant.constant as var2,
        sampling_time_step_count,
    from Model
    join FlowMatchingMarginal on Model.id = FlowMatchingMarginal.id
    join PreviousPosteriorToPredictive on FlowMatchingMarginal.DiffusionPath = PreviousPosteriorToPredictive.id
    join Local on FlowMatchingMarginal.EnergyGuidance = Local.id
    join Constant on Local.Schedule = Constant.id
    join hyperparameters on (
            hyperparameters.label = {label!r}
        and
            (hyperparameters.sigma_min, hyperparameters.lambda) = (PreviousPosteriorToPredictive.sigma_min, Constant.constant)
    )
    where true
    and sampling_time_step_count in {sampling_time_step_counts}
""")
enff_f2p_multiple = (
    len(sampling_time_step_counts)
    * len(hyperparameters.filter(label=label)[['sigma_min', 'lambda']].unique())
)
assert len(enff_f2p_rows) == enff_f2p_multiple

### General

In [80]:
rows = duckdb.sql(f"""
    select
        alt_id,
        model_rows.*,
        dataset_rows.*,
        rng_seed,
    from Conf
    join (
        select * from l96_rows
        union
        select * from ks_rows
        union
        select * from ns64_rows
        union
        select * from ns256_rows
    ) as dataset_rows on Conf.Dataset = dataset_rows.id
    join (
        select * from ensf_rows
        union
        select * from enff_ot_rows
        union
        select * from enff_f2p_rows
    ) as model_rows on Conf.Model = model_rows.id
    where true
    and rng_seed in (462133975, 979497033, 97616566, 715319214, 19704671)
""")
rng_seed_multiple = 5
dataset_multiple = dataset_info['multiple'].sum()
model_multiple = 3
multiple = rng_seed_multiple * dataset_multiple * model_multiple * len(sampling_time_step_counts)
assert len(rows) == multiple, f'{len(rows) = } != {multiple}'

In [81]:
filepaths = duckdb.sql("""
select format('~/out/dafm/runs/{}/dataset_metrics.csv', alt_id) as filepath from rows
""").pl()
filepath_exists = []
for f in filepaths.get_column('filepath'):
    f = Path(f).expanduser()
    exists = f.exists()
    if not exists:
        print(f"File path does not exist: '{f}'")
    filepath_exists.append(exists)
filepaths = pl.DataFrame(dict(filepath=filepaths.get_column('filepath'), exists=filepath_exists))
assert filepaths['exists'].any()
duckdb.sql("""
set variable dataset_metrics_filepaths = (
    select list(filepath) from filepaths where exists
)
""")

File path does not exist: '/home/ttransue/out/dafm/runs/yxz1lyxm/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/565k2wdh/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/0ukgql19/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/z1wxlbsw/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/au48lv7j/dataset_metrics.csv'


##### Time (seconds)

In [82]:
logged_metrics = duckdb.sql(f"""
    select rows.*, logs.*,
    from (
        select split(filename, '/')[-2] as alt_id, step, time_s, crps, rmse,
        from read_csv(getvariable(dataset_metrics_filepaths), filename=true, union_by_name=true)
    ) as logs
    join rows on rows.alt_id = logs.alt_id
    where true
    and (logs.step - time_step_count_drop_first - 1) % observe_every_n_time_steps == 0 -- include only analysis time steps
""")
logged_metrics.show(max_width=100)

┌──────────┬───────┬──────────┬───┬────────────────────┬──────────────────────┐
│  alt_id  │  id   │  label   │ … │        crps        │         rmse         │
│ varchar  │ int64 │ varchar  │   │       double       │        double        │
├──────────┼───────┼──────────┼───┼────────────────────┼──────────────────────┤
│ elt43fpj │   581 │ EnFF-F2P │ … │ 1.3592673803856108 │  0.04341920833932852 │
│ 0xmn5krf │   922 │ EnFF-OT  │ … │ 2.4173822804927925 │  0.07554790055032799 │
│ 3n5cnu1j │   665 │ EnFF-OT  │ … │  2.434296458385194 │  0.07607727913047352 │
│ o8tvo2vd │   923 │ EnFF-OT  │ … │ 37.109436543266106 │   1.1596761315166992 │
│ 60v6qosl │   924 │ EnFF-OT  │ … │ 51.065203948430415 │    1.595794203961202 │
│ e729q8dl │  1519 │ EnFF-OT  │ … │  64.77322018228058 │   2.0241698137054844 │
│ 5seil1rl │   627 │ EnFF-F2P │ … │  8.267454052900485 │  0.25934018211275567 │
│ 5wgy6hrx │   628 │ EnFF-F2P │ … │    8.6885202027527 │   0.2726257960894323 │
│ t2sdnvdr │   629 │ EnFF-F2P │ … │  8.8

In [83]:
max_observation_steps_back = duckdb.sql(f"""
    select
        dataset_name,
        max(observation_step_count) as observation_step_count,
    from (
        select
            dataset_name,
            count(*) as observation_step_count,
        from logged_metrics
        group by alt_id, dataset_name
    )
    group by dataset_name
""")
failed_before_finish = duckdb.sql(f"""
    select
        observation_steps_back.alt_id,
        observation_steps_back.dataset_name,
        observation_steps_back.label,
        observation_steps_back.observation_step_count,
    from (
        select
            alt_id,
            dataset_name,
            label,
            count(*) as observation_step_count,
        from logged_metrics
        group by alt_id, dataset_name, label
    ) as observation_steps_back
    join max_observation_steps_back
    on max_observation_steps_back.dataset_name = observation_steps_back.dataset_name
    and observation_steps_back.observation_step_count < max_observation_steps_back.observation_step_count
""")
# max_observation_steps_back
failed_before_finish
# assert len(logged_metrics) > 0 and len(failed_before_finish) == 0, failed_before_finish

┌──────────┬────────────────────┬─────────┬────────────────────────┐
│  alt_id  │    dataset_name    │  label  │ observation_step_count │
│ varchar  │      varchar       │ varchar │         int64          │
├──────────┼────────────────────┼─────────┼────────────────────────┤
│ 43xm7nny │ NavierStokesDim64  │ EnFF-OT │                     20 │
│ q2cb15pc │ NavierStokesDim256 │ EnSF    │                      1 │
│ 67ibhkaq │ NavierStokesDim64  │ EnFF-OT │                     10 │
│ 7rsvcb7m │ NavierStokesDim64  │ EnFF-OT │                     10 │
│ zhikn89j │ NavierStokesDim256 │ EnSF    │                      1 │
│ v86rvuwe │ NavierStokesDim256 │ EnSF    │                      1 │
│ 46rzlpry │ NavierStokesDim64  │ EnFF-OT │                     20 │
│ iqki912t │ NavierStokesDim64  │ EnFF-OT │                     20 │
│ i49fmdh2 │ NavierStokesDim64  │ EnFF-OT │                     20 │
│ dbp9nsl8 │ NavierStokesDim256 │ EnSF    │                      1 │
│ 7ihuxpfn │ NavierStokesDim256 │ 

In [84]:
logged_metrics = duckdb.sql("""
    select *
    from logged_metrics
    where alt_id not in (select alt_id from failed_before_finish)
""")

In [85]:
group_by = """
    alt_id,
    rng_seed,
    dataset_name,
    dataset_name_latex,
    label,
    sampling_time_step_count,
"""
logged_metrics_means = duckdb.sql(f"""
    select
        {group_by}
        mean(time_s) as time_s_mean,
        mean(rmse) as rmse_mean,
        mean(crps) as crps_mean,
    from logged_metrics
    group by
        {group_by}
""")
logged_metrics_means.show(max_width=100)

┌──────────┬───────────┬───┬──────────────────────┬──────────────────────┬────────────────────┐
│  alt_id  │ rng_seed  │ … │     time_s_mean      │      rmse_mean       │     crps_mean      │
│ varchar  │   int64   │   │        double        │        double        │       double       │
├──────────┼───────────┼───┼──────────────────────┼──────────────────────┼────────────────────┤
│ rlure1rf │ 715319214 │ … │  0.05651241107499981 │   0.3307450021803379 │ 32.034897003173825 │
│ xza2iz8h │ 715319214 │ … │    5.018023847100002 │   0.3346771389245987 │ 331.55544033050535 │
│ 6k2w53xv │  97616566 │ … │  0.19794395626100017 │  0.06270453071920509 │  1.891008599717045 │
│ kqmqrmo5 │ 715319214 │ … │  0.01677092336899998 │  0.05650676458820532 │ 1.8029669223440508 │
│ iij4i185 │ 462133975 │ … │ 0.027777049445000025 │  0.06576520655304194 │  7.547560040950775 │
│ fcii7zj6 │  97616566 │ … │  0.03194046320500044 │  0.17239961240440607 │ 19.110407366752625 │
│ d0bhboze │  19704671 │ … │   1.1714295

In [86]:
group_by = """
    dataset_name,
    dataset_name_latex,
    label,
    sampling_time_step_count,
"""
table_stats = duckdb.sql(f"""
    select
        {group_by}
        mean(time_s_mean) as time_s_mean,
        stddev_pop(time_s_mean) as time_s_std,
        mean(rmse_mean) as rmse_mean,
        stddev_pop(rmse_mean) as rmse_std,
    from logged_metrics_means
    group by
        {group_by}
""")
table_stats.show(max_width=100)

┌─────────────────────┬──────────────────────┬───┬─────────────────────┬──────────────────────┐
│    dataset_name     │  dataset_name_latex  │ … │      rmse_mean      │       rmse_std       │
│       varchar       │       varchar        │   │       double        │        double        │
├─────────────────────┼──────────────────────┼───┼─────────────────────┼──────────────────────┤
│ Lorenz96            │ Lorenz '96           │ … │  3.3001568609476086 │ 0.009263572826161488 │
│ Lorenz96            │ Lorenz '96           │ … │  0.3334900294989347 │  0.0004277596978527… │
│ NavierStokesDim256  │ Navier-Stokes ($25…  │ … │ 0.45672727265084784 │ 0.049823500715084665 │
│ NavierStokesDim64   │ Navier-Stokes ($64…  │ … │ 0.06700994930788876 │  0.0007031539072056… │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ … │ 0.19993277289814781 │  0.0003163244249420… │
│ Lorenz96            │ Lorenz '96           │ … │  0.6600737687200308 │  0.0032300217032167… │
│ NavierStokesDim256  │ Navier-Stokes ($

In [87]:
table_stats = duckdb.sql(f"""
    select
        *,
        time_s_mean = min(time_s_mean) over (partition by dataset_name, sampling_time_step_count) as is_lowest_time,
        rmse_mean = min(rmse_mean) over (partition by dataset_name, sampling_time_step_count) as is_lowest_rmse,
    from table_stats
""")
table_stats.show(max_width=100)

┌─────────────────────┬──────────────────────┬──────────┬───┬────────────────┬────────────────┐
│    dataset_name     │  dataset_name_latex  │  label   │ … │ is_lowest_time │ is_lowest_rmse │
│       varchar       │       varchar        │ varchar  │   │    boolean     │    boolean     │
├─────────────────────┼──────────────────────┼──────────┼───┼────────────────┼────────────────┤
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ EnSF     │ … │ false          │ false          │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ EnFF-F2P │ … │ false          │ false          │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ EnFF-OT  │ … │ true           │ true           │
│ NavierStokesDim256  │ Navier-Stokes ($25…  │ EnSF     │ … │ false          │ false          │
│ NavierStokesDim256  │ Navier-Stokes ($25…  │ EnFF-OT  │ … │ true           │ false          │
│ NavierStokesDim256  │ Navier-Stokes ($25…  │ EnFF-F2P │ … │ false          │ true           │
│ NavierStokesDim64   │ Navier-Stokes ($

In [88]:
def join(t):
    if t != '':
        t = f'{t}.'
    return f'{t}dataset_name, {t}sampling_time_step_count, {t}label'
table = duckdb.sql(rf"""
    select
        table_stats.label as 'Model',
        table_stats.dataset_name_latex as 'System',
        table_stats.sampling_time_step_count as '$T$',
        if(is_lowest_time, format('$\bm{{{{{{}}}}}}$', fmt_stats.time_s), format('${{}}$', fmt_stats.time_s)) as 'Time (s)',
        if(is_lowest_rmse, format('$\bm{{{{{{}}}}}}$', fmt_stats.rmse), format('${{}}$', fmt_stats.rmse)) as 'RMSE',
    from table_stats
    join (
        select
            {join('')},
            format('{{{{{{:.3f}}}}}}_{{{{\pm {{:.3f}}}}}}', time_s_mean, time_s_std) as time_s,
            format('{{{{{{:.3f}}}}}}_{{{{\pm {{:.3f}}}}}}', rmse_mean, rmse_mean) as rmse,
        from table_stats
    ) as fmt_stats
    on ({join('table_stats')}) = ({join('fmt_stats')})
    order by
        {join('table_stats')} desc
""")
table

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────────────────────────┬───────┬────────────────────────────┬────────────────────────────┐
│  Model   │              System              │  $T$  │          Time (s)          │            RMSE            │
│ varchar  │             varchar              │ int64 │          varchar           │          varchar           │
├──────────┼──────────────────────────────────┼───────┼────────────────────────────┼────────────────────────────┤
│ EnSF     │ Kuramoto-Sivashinsky (1,024 dim) │     5 │ ${0.010}_{\pm 0.000}$      │ ${1.631}_{\pm 1.631}$      │
│ EnFF-OT  │ Kuramoto-Sivashinsky (1,024 dim) │     5 │ $\bm{{0.008}_{\pm 0.000}}$ │ ${0.082}_{\pm 0.082}$      │
│ EnFF-F2P │ Kuramoto-Sivashinsky (1,024 dim) │     5 │ ${0.008}_{\pm 0.000}$      │ $\bm{{0.065}_{\pm 0.065}}$ │
│ EnSF     │ Kuramoto-Sivashinsky (1,024 dim) │    10 │ ${0.019}_{\pm 0.000}$      │ ${0.640}_{\pm 0.640}$      │
│ EnFF-OT  │ Kuramoto-Sivashinsky (1,024 dim) │    10 │ $\bm{{0.017}_{\pm 0.001}}$ │ $\b

In [89]:
table_pl = table.pl()
table_pl_pivot = table_pl.pivot(
    on=['System'],
    index=['Model', '$T$'],
    values=['Time (s)', 'RMSE'],
    separator='|',
).fill_null(r'\texttt{NaN}')
cols = ['Model', '$T$']
for system in dataset_info['dataset_name_latex']:
    cols.extend([f'Time (s)|{system}', f'RMSE|{system}'])
table_pl_pivot = table_pl_pivot[cols]
gt = GT(table_pl_pivot)
gt = gt.tab_spanner(label=' ', columns=['Model', '$T$'])
# gt = gt.tab_stub(rowname_col='Model', groupname_col='$T$')
for system in dataset_info['dataset_name_latex']:
    cols = [f'Time (s)|{system}', f'RMSE|{system}']
    gt = gt.tab_spanner(label=system, columns=cols).cols_label(**{s: s.split('|')[0] for s in cols})
gt

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GT(_tbl_data=shape: (15, 10)
┌──────────┬─────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ Model    ┆ $T$ ┆ Time       ┆ RMSE|Loren ┆ … ┆ Time (s)|N ┆ RMSE|Navie ┆ Time (s)|N ┆ RMSE|Navie │
│ ---      ┆ --- ┆ (s)|Lorenz ┆ z '96      ┆   ┆ avier-Stok ┆ r-Stokes   ┆ avier-Stok ┆ r-Stokes   │
│ str      ┆ i64 ┆ '96        ┆ ---        ┆   ┆ es ($64    ┆ ($64       ┆ es ($256   ┆ ($256      │
│          ┆     ┆ ---        ┆ str        ┆   ┆ \t…        ┆ \times…    ┆ \…         ┆ \time…     │
│          ┆     ┆ str        ┆            ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│          ┆     ┆            ┆            ┆   ┆ str        ┆ str        ┆ str        ┆ str        │
╞══════════╪═════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ EnSF     ┆ 5   ┆ $\bm{{0.44 ┆ ${7.666}_{ ┆ … ┆ \texttt{Na ┆ \texttt{Na ┆ \texttt{Na ┆ \texttt{Na │
│          ┆     ┆ 4}_{\pm    ┆ \pm        ┆   ┆ N}         ┆ N}         ┆ N}         ┆ N}         │
│          ┆     ┆ 0.034}}$   ┆ 7.666}$    ┆   ┆            ┆            ┆            ┆            │
│ EnFF-OT  ┆ 5   ┆ ${0.470}_{ ┆ $\bm{{0.62 ┆ … ┆ ${0.026}_{ ┆ ${0.164}_{ ┆ $\bm{{0.19 ┆ ${0.494}_{ │
│          ┆     ┆ \pm        ┆ 3}_{\pm    ┆   ┆ \pm        ┆ \pm        ┆ 0}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.016}$    ┆ 0.623}}$   ┆   ┆ 0.004}$    ┆ 0.164}$    ┆ 0.008}}$   ┆ 0.494}$    │
│ EnFF-F2P ┆ 5   ┆ ${0.473}_{ ┆ ${0.649}_{ ┆ … ┆ $\bm{{0.02 ┆ $\bm{{0.06 ┆ ${0.191}_{ ┆ $\bm{{0.13 │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ 4}_{\pm    ┆ 7}_{\pm    ┆ \pm        ┆ 2}_{\pm    │
│          ┆     ┆ 0.020}$    ┆ 0.649}$    ┆   ┆ 0.002}}$   ┆ 0.067}}$   ┆ 0.009}$    ┆ 0.132}}$   │
│ EnSF     ┆ 10  ┆ $\bm{{0.67 ┆ ${3.300}_{ ┆ … ┆ ${0.035}_{ ┆ ${1.548}_{ ┆ ${0.258}_{ ┆ ${5.403}_{ │
│          ┆     ┆ 8}_{\pm    ┆ \pm        ┆   ┆ \pm        ┆ \pm        ┆ \pm        ┆ \pm        │
│          ┆     ┆ 0.026}}$   ┆ 3.300}$    ┆   ┆ 0.001}$    ┆ 1.548}$    ┆ 0.023}$    ┆ 5.403}$    │
│ EnFF-OT  ┆ 10  ┆ ${0.707}_{ ┆ ${0.581}_{ ┆ … ┆ $\bm{{0.03 ┆ ${0.112}_{ ┆ $\bm{{0.22 ┆ ${0.497}_{ │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ 1}_{\pm    ┆ \pm        ┆ 9}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.006}$    ┆ 0.581}$    ┆   ┆ 0.001}}$   ┆ 0.112}$    ┆ 0.012}}$   ┆ 0.497}$    │
│ …        ┆ …   ┆ …          ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ EnFF-OT  ┆ 50  ┆ ${2.579}_{ ┆ ${0.704}_{ ┆ … ┆ $\bm{{0.10 ┆ ${0.129}_{ ┆ $\bm{{0.56 ┆ ${0.373}_{ │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ 1}_{\pm    ┆ \pm        ┆ 1}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.012}$    ┆ 0.704}$    ┆   ┆ 0.004}}$   ┆ 0.129}$    ┆ 0.013}}$   ┆ 0.373}$    │
│ EnFF-F2P ┆ 50  ┆ ${2.628}_{ ┆ $\bm{{0.33 ┆ … ┆ ${0.104}_{ ┆ $\bm{{0.06 ┆ ${0.583}_{ ┆ $\bm{{0.13 │
│          ┆     ┆ \pm        ┆ 3}_{\pm    ┆   ┆ \pm        ┆ 6}_{\pm    ┆ \pm        ┆ 2}_{\pm    │
│          ┆     ┆ 0.011}$    ┆ 0.333}}$   ┆   ┆ 0.003}$    ┆ 0.066}}$   ┆ 0.024}$    ┆ 0.132}}$   │
│ EnSF     ┆ 100 ┆ $\bm{{4.88 ┆ ${1.374}_{ ┆ … ┆ ${0.230}_{ ┆ ${0.218}_{ ┆ ${1.077}_{ ┆ ${0.900}_{ │
│          ┆     ┆ 1}_{\pm    ┆ \pm        ┆   ┆ \pm        ┆ \pm        ┆ \pm        ┆ \pm        │
│          ┆     ┆ 0.025}}$   ┆ 1.374}$    ┆   ┆ 0.006}$    ┆ 0.218}$    ┆ 0.035}$    ┆ 0.900}$    │
│ EnFF-OT  ┆ 100 ┆ ${4.923}_{ ┆ ${0.719}_{ ┆ … ┆ \texttt{Na ┆ \texttt{Na ┆ $\bm{{0.97 ┆ ${0.270}_{ │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ N}         ┆ N}         ┆ 6}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.016}$    ┆ 0.719}$    ┆   ┆            ┆            ┆ 0.016}}$   ┆ 0.270}$    │
│ EnFF-F2P ┆ 100 ┆ ${5.020}_{ ┆ $\bm{{0.33 ┆ … ┆ $\bm{{0.19 ┆ $\bm{{0.06 ┆ ${1.045}_{ ┆ $\bm{{0.13 │
│          ┆     ┆ \pm        ┆ 4}_{\pm    ┆   ┆ 5}_{\pm    ┆ 6}_{\pm    ┆ \pm        ┆ 2}_{\pm    │
│          ┆     ┆ 0.016}$    ┆ 0.334}}$   ┆   ┆ 0.004}}$   ┆ 0.066}}$   ┆ 0.085}$    ┆ 0.132}}$   │
└──────────┴─────┴────────────┴─

In [90]:
table_latex = (
    gt.as_latex()
    .replace(r'\{', r'{')
    .replace(r'\}', r'}')
    .replace(r'\_', r'_')
    .replace(r'\$', r'$')
    .replace(r'\\pm', r'\pm')
    .replace(r'\\bm', r'\bm')
    .replace(r'\\times', r'\times')
    .replace(r'\\texttt', r'\texttt')
    .replace('Kuramoto-Sivashinsky', 'KS')
    .replace('Navier-Stokes', 'NS')
    .replace('EnFF-OT', r'EnFF-OT$^\ast$')
    .replace('EnFF-F2P', r'EnFF-F2P$^\ast$')
)
print(table_latex)

\begin{table}[!t]


\fontsize{12.0pt}{14.4pt}\selectfont

\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}}lrllllllll}
\toprule
\multicolumn{2}{c}{ } & \multicolumn{2}{c}{Lorenz '96} & \multicolumn{2}{c}{KS (1,024 dim)} & \multicolumn{2}{c}{NS ($64 \times 64$)} & \multicolumn{2}{c}{NS ($256 \times 256$)} \\ 
\cmidrule(lr){1-2} \cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8} \cmidrule(lr){9-10}
Model & $T$ & Time (s) & RMSE & Time (s) & RMSE & Time (s) & RMSE & Time (s) & RMSE \\ 
\midrule\addlinespace[2.5pt]
EnSF & 5 & $\bm{{0.444}_{\pm 0.034}}$ & ${7.666}_{\pm 7.666}$ & ${0.010}_{\pm 0.000}$ & ${1.631}_{\pm 1.631}$ & \texttt{NaN} & \texttt{NaN} & \texttt{NaN} & \texttt{NaN} \\
EnFF-OT$^\ast$ & 5 & ${0.470}_{\pm 0.016}$ & $\bm{{0.623}_{\pm 0.623}}$ & $\bm{{0.008}_{\pm 0.000}}$ & ${0.082}_{\pm 0.082}$ & ${0.026}_{\pm 0.004}$ & ${0.164}_{\pm 0.164}$ & $\bm{{0.190}_{\pm 0.008}}$ & ${0.494}_{\pm 0.494}$ \\
EnFF-F2P$^\ast$ & 5 & ${0.473}_{\pm 0.020}$ & ${0.649}_{\pm 0.649}$ & ${0